In [1]:
# ============================================================
# Q3 — Correct Imputation vs Data Leakage
# ============================================================

import numpy as np
import pandas as pd

from sklearn.datasets import fetch_california_housing

from sklearn.model_selection import KFold, cross_val_score

from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression


# ------------------------------------------------------------
# 1. Load dataset
# ------------------------------------------------------------

housing = fetch_california_housing(as_frame=True)

X = housing.data.copy()
y = housing.target.copy()




In [2]:
# ------------------------------------------------------------
# 2. Inject missing values
# ------------------------------------------------------------

rng = np.random.RandomState(42)

X_missing = X.copy()

missing_rate = 0.20

mask = rng.rand(*X_missing.shape) < missing_rate

X_missing[mask] = np.nan


print("Missing percentage:")
print(X_missing.isnull().mean() * 100)



Missing percentage:
MedInc        19.815891
HouseAge      20.445736
AveRooms      20.188953
AveBedrms     19.941860
Population    20.174419
AveOccup      19.748062
Latitude      19.641473
Longitude     20.193798
dtype: float64


In [3]:
 #------------------------------------------------------------
# 3. Define identical CV folds
# ------------------------------------------------------------

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


# ============================================================
# CORRECT APPROACH
# ============================================================

# ------------------------------------------------------------
# 4. Imputer INSIDE Pipeline
# ------------------------------------------------------------

correct_pipeline = Pipeline([
    
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    
    (
        "model",
        LinearRegression()
    )
])

In [4]:
# ------------------------------------------------------------
# 5. Cross-validation
# ------------------------------------------------------------

correct_scores = cross_val_score(
    correct_pipeline,
    X_missing,
    y,
    cv=cv,
    scoring="neg_root_mean_squared_error"
)


# Convert negative RMSE to positive RMSE

correct_rmse = -correct_scores


print("\nCorrect Pipeline CV RMSE:")
print(correct_rmse)

print(
    "\nMean Correct RMSE:",
    correct_rmse.mean()
)



Correct Pipeline CV RMSE:
[0.887304   0.88664166 0.86752011 0.86453855 0.94667742]

Mean Correct RMSE: 0.8905363481909045


In [5]:
# ============================================================
# LEAKED APPROACH
# ============================================================

# ------------------------------------------------------------
# 6. Fit imputer on the ENTIRE dataset
# ------------------------------------------------------------

leaked_imputer = SimpleImputer(
    strategy="median"
)

X_imputed = leaked_imputer.fit_transform(
    X_missing
)


# ------------------------------------------------------------
# 7. Cross-validation AFTER full-data imputation
# ------------------------------------------------------------

leaked_model = LinearRegression()

leaked_scores = cross_val_score(
    leaked_model,
    X_imputed,
    y,
    cv=cv,
    scoring="neg_root_mean_squared_error"
)



In [6]:
# Convert to positive RMSE

leaked_rmse = -leaked_scores


print("\nLeaked CV RMSE:")
print(leaked_rmse)

print(
    "\nMean Leaked RMSE:",
    leaked_rmse.mean()
)


# ============================================================
# 8. Compare
# ============================================================

comparison_q3 = pd.DataFrame({
    
    "Approach": [
        "Correct Pipeline",
        "Leaked Imputation"
    ],
    
    "Mean CV RMSE": [
        correct_rmse.mean(),
        leaked_rmse.mean()
    ]
})

print("\nComparison:")
display(comparison_q3)


Leaked CV RMSE:
[0.88741685 0.88651336 0.86738046 0.86461235 0.94670051]

Mean Leaked RMSE: 0.8905247074704713

Comparison:


,Approach,Mean CV RMSE
0,Correct Pipeline,0.890536
1,Leaked Imputation,0.890525


In [7]:
# ============================================================
# 9. Calculate optimistic bias
# ============================================================

optimistic_bias = (
    correct_rmse.mean()
    -
    leaked_rmse.mean()
)

print(
    "\nOptimistic Bias:",
    optimistic_bias
)


# ------------------------------------------------------------
# 10. Percentage optimistic bias
# ------------------------------------------------------------

percentage_bias = (
    optimistic_bias
    /
    correct_rmse.mean()
) * 100

print(
    "Optimistic Bias (%):",
    percentage_bias
)


Optimistic Bias: 1.1640720433248042e-05
Optimistic Bias (%): 0.0013071583722433996
